# 🏠 House Price Prediction using Machine Learning
**IBM SkillsBuild Internship Project**

This notebook covers the complete machine-learning pipeline:

1. Load & understand the dataset
2. Exploratory Data Analysis (EDA)
3. Feature engineering & preprocessing
4. Training and comparing multiple regression models
5. Hyper-parameter tuning
6. Evaluation & interpretation
7. Saving the trained model (used by the Streamlit frontend `app.py`)

**Backend:** Python, pandas, NumPy, scikit-learn  |  **Frontend:** Streamlit + Matplotlib (see `app.py`)

## 1. Import Libraries

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

RANDOM_STATE = 42
REFERENCE_YEAR = 2026          # used to compute House_Age
DATA_PATH = "house_price_regression_dataset.csv"
os.makedirs("models", exist_ok=True)
os.makedirs("figures", exist_ok=True)
print("Libraries imported successfully ✅")

## 2. Load the Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape of dataset (rows, columns):", df.shape)
df.head()

### Dataset description
| Column | Meaning |
|---|---|
| Square_Footage | Built-up area of the house (sq ft) |
| Num_Bedrooms | Number of bedrooms |
| Num_Bathrooms | Number of bathrooms |
| Year_Built | Year the house was built |
| Lot_Size | Size of the plot/lot (acres) |
| Garage_Size | Number of garage spaces |
| Neighborhood_Quality | Neighbourhood rating (1 = low, 10 = high) |
| **House_Price** | **Target variable – price of the house** |

## 3. Data Understanding & Quality Check

In [ ]:
df.info()

In [ ]:
df.describe().T.round(2)

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

✅ The dataset has **no missing values and no duplicates**, so no imputation or row removal is needed.

## 4. Exploratory Data Analysis (EDA)

### 4.1 Distribution of the target (House Price)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df["House_Price"], kde=True, ax=axes[0], color="#2E75B6")
axes[0].set_title("Distribution of House Price")
axes[0].set_xlabel("House Price")
sns.boxplot(x=df["House_Price"], ax=axes[1], color="#9DC3E6")
axes[1].set_title("Box Plot of House Price")
plt.tight_layout()
plt.savefig("figures/01_price_distribution.png", bbox_inches="tight")
plt.show()
print("Mean price  :", round(df['House_Price'].mean(), 2))
print("Median price:", round(df['House_Price'].median(), 2))
print("Skewness    :", round(df['House_Price'].skew(), 3))

### 4.2 Distribution of all numerical features

In [ ]:
feature_cols = [c for c in df.columns if c != "House_Price"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), feature_cols):
    sns.histplot(df[col], kde=False, ax=ax, color="#2E75B6", bins=20)
    ax.set_title(col)
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.savefig("figures/02_feature_distributions.png", bbox_inches="tight")
plt.show()

### 4.3 Correlation analysis

In [ ]:
plt.figure(figsize=(9, 7))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True, linewidths=.5)
plt.title("Correlation Heatmap")
plt.savefig("figures/03_correlation_heatmap.png", bbox_inches="tight")
plt.show()

print("Correlation of every feature with House_Price:")
print(corr["House_Price"].drop("House_Price").sort_values(ascending=False).round(3))

**Insight:** `Square_Footage` has an extremely strong positive correlation with price (~0.99), so house size is by far the most important driver. `Lot_Size` shows a weak positive relationship; other features are weak individually.

### 4.4 Square footage vs price

In [ ]:
plt.figure(figsize=(9, 5.5))
sns.scatterplot(data=df, x="Square_Footage", y="House_Price", hue="Neighborhood_Quality",
                palette="viridis", alpha=.8)
sns.regplot(data=df, x="Square_Footage", y="House_Price", scatter=False, color="red",
            line_kws={"linewidth": 2})
plt.title("Square Footage vs House Price")
plt.savefig("figures/04_sqft_vs_price.png", bbox_inches="tight")
plt.show()

### 4.5 Price by bedrooms, bathrooms, garage and neighbourhood quality

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.ravel(), ["Num_Bedrooms", "Num_Bathrooms", "Garage_Size", "Neighborhood_Quality"]):
    sns.boxplot(data=df, x=col, y="House_Price", ax=ax, palette="Blues")
    ax.set_title(f"House Price by {col}")
plt.tight_layout()
plt.savefig("figures/05_price_by_categories.png", bbox_inches="tight")
plt.show()

### 4.6 Price trend by decade built

In [ ]:
tmp = df.copy()
tmp["Decade"] = (tmp["Year_Built"] // 10 * 10).astype(int)
decade_avg = tmp.groupby("Decade")["House_Price"].mean()
plt.figure(figsize=(10, 4.5))
decade_avg.plot(kind="bar", color="#2E75B6")
plt.title("Average House Price by Decade Built")
plt.ylabel("Average price")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("figures/06_price_by_decade.png", bbox_inches="tight")
plt.show()

## 5. Feature Engineering & Preprocessing
* Create **House_Age** = `REFERENCE_YEAR − Year_Built` (age is easier for a model to interpret than a raw year).
* Drop `Year_Built` (it carries the same information as House_Age).
* Use an **80 / 20 train-test split**.
* Scale numerical features with `StandardScaler` **inside a Pipeline** so there is no data leakage.

In [ ]:
data = df.copy()
data["House_Age"] = REFERENCE_YEAR - data["Year_Built"]
data = data.drop(columns=["Year_Built"])

FEATURES = ["Square_Footage", "Num_Bedrooms", "Num_Bathrooms", "Lot_Size",
            "Garage_Size", "Neighborhood_Quality", "House_Age"]
TARGET = "House_Price"

X = data[FEATURES]
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Training set:", X_train.shape, "| Test set:", X_test.shape)

## 6. Model Building & Comparison
We compare six regression algorithms using **5-fold cross-validation** on the training set.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, random_state=RANDOM_STATE),
}

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, model in models.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
    cv_r2 = cross_val_score(pipe, X_train, y_train, cv=kf, scoring="r2")
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({
        "Model": name,
        "CV R2 (mean)": cv_r2.mean(),
        "Test R2": r2_score(y_test, pred),
        "Test MAE": mean_absolute_error(y_test, pred),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, pred)),
    })

results = pd.DataFrame(rows).sort_values("Test R2", ascending=False).reset_index(drop=True)
results.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sns.barplot(data=results, x="Test R2", y="Model", ax=axes[0], palette="Blues_r")
axes[0].set_title("Model Comparison – Test R² Score")
axes[0].set_xlim(0, 1.05)
sns.barplot(data=results, x="Test RMSE", y="Model", ax=axes[1], palette="Oranges_r")
axes[1].set_title("Model Comparison – Test RMSE (lower is better)")
plt.tight_layout()
plt.savefig("figures/07_model_comparison.png", bbox_inches="tight")
plt.show()

## 7. Hyper-parameter Tuning
We tune the two strongest candidate families (Ridge and Gradient Boosting) with `GridSearchCV`.

In [ ]:
ridge_grid = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
    {"model__alpha": [0.01, 0.1, 1, 10, 100]},
    cv=kf, scoring="r2", n_jobs=-1)
ridge_grid.fit(X_train, y_train)

gb_grid = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("model", GradientBoostingRegressor(random_state=RANDOM_STATE))]),
    {"model__n_estimators": [100, 300],
     "model__learning_rate": [0.05, 0.1],
     "model__max_depth": [2, 3]},
    cv=kf, scoring="r2", n_jobs=-1)
gb_grid.fit(X_train, y_train)

print("Best Ridge params           :", ridge_grid.best_params_, "| CV R2 =", round(ridge_grid.best_score_, 4))
print("Best Gradient Boosting params:", gb_grid.best_params_, "| CV R2 =", round(gb_grid.best_score_, 4))

## 8. Select the Final Model & Evaluate on the Test Set

In [ ]:
candidates = {"Ridge Regression (tuned)": ridge_grid.best_estimator_,
              "Gradient Boosting (tuned)": gb_grid.best_estimator_}

final_scores = {}
for name, est in candidates.items():
    p = est.predict(X_test)
    final_scores[name] = {"R2": r2_score(y_test, p),
                          "MAE": mean_absolute_error(y_test, p),
                          "RMSE": float(np.sqrt(mean_squared_error(y_test, p)))}
pd.DataFrame(final_scores).T.round(4)

In [ ]:
best_name = max(final_scores, key=lambda k: final_scores[k]["R2"])
best_model = candidates[best_name]
y_pred = best_model.predict(X_test)

metrics = {
    "best_model": best_name,
    "r2": float(r2_score(y_test, y_pred)),
    "mae": float(mean_absolute_error(y_test, y_pred)),
    "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
    "n_train": int(len(X_train)), "n_test": int(len(X_test)),
    "comparison": results.round(4).to_dict(orient="records"),
}
print(f"🏆 Final model : {best_name}")
print(f"R² Score       : {metrics['r2']:.4f}")
print(f"MAE            : {metrics['mae']:,.2f}")
print(f"RMSE           : {metrics['rmse']:,.2f}")

### 8.1 Actual vs Predicted & Residual analysis

In [ ]:
residuals = y_test - y_pred
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

axes[0].scatter(y_test, y_pred, alpha=.6, color="#2E75B6")
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, "r--", label="Perfect prediction")
axes[0].set_xlabel("Actual price"); axes[0].set_ylabel("Predicted price")
axes[0].set_title("Actual vs Predicted"); axes[0].legend()

axes[1].scatter(y_pred, residuals, alpha=.6, color="#C55A11")
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_xlabel("Predicted price"); axes[1].set_ylabel("Residual")
axes[1].set_title("Residual Plot")

sns.histplot(residuals, kde=True, ax=axes[2], color="#548235")
axes[2].set_title("Residual Distribution")
plt.tight_layout()
plt.savefig("figures/08_actual_vs_predicted.png", bbox_inches="tight")
plt.show()

### 8.2 Feature importance (permutation importance)

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, n_repeats=15,
                              random_state=RANDOM_STATE, scoring="r2")
importance = pd.Series(perm.importances_mean, index=FEATURES).sort_values()

plt.figure(figsize=(9, 5))
importance.plot(kind="barh", color="#2E75B6")
plt.title("Feature Importance (drop in R² when feature is shuffled)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("figures/09_feature_importance.png", bbox_inches="tight")
plt.show()

metrics["feature_importance"] = {k: float(v) for k, v in importance.sort_values(ascending=False).items()}
importance.sort_values(ascending=False).round(4)

## 9. Save the Model for the Frontend

In [ ]:
joblib.dump({"model": best_model, "features": FEATURES, "reference_year": REFERENCE_YEAR},
            "models/house_price_model.joblib")
with open("models/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved: models/house_price_model.joblib and models/metrics.json ✅")

## 10. Test a Sample Prediction

In [ ]:
def predict_price(sqft, bedrooms, bathrooms, year_built, lot_size, garage, neighborhood_quality):
    row = pd.DataFrame([{
        "Square_Footage": sqft, "Num_Bedrooms": bedrooms, "Num_Bathrooms": bathrooms,
        "Lot_Size": lot_size, "Garage_Size": garage,
        "Neighborhood_Quality": neighborhood_quality,
        "House_Age": REFERENCE_YEAR - year_built}])[FEATURES]
    return float(best_model.predict(row)[0])

sample = predict_price(sqft=2500, bedrooms=3, bathrooms=2, year_built=2005,
                       lot_size=2.5, garage=1, neighborhood_quality=7)
print(f"Estimated price for the sample house: {sample:,.2f}")

## 11. Conclusion
* The dataset is clean (1000 rows, 7 features, no missing values).
* **Square footage** is the dominant price driver; other features contribute marginally.
* Several algorithms were compared with cross-validation and the best tuned model was selected on the hold-out test set.
* The trained model is saved and served through a **Streamlit web app** (`app.py`) for interactive price prediction.

**Run the frontend:** `streamlit run app.py`